In [ ]:

# Le but de ce script est de développer une intuition graphique
# pour l'accélération de Coriolis. Nous allons commencer
# par tracer un repère de référence avec des vecteurs appelés I et J (selon x et y)
# que nous allons animer d'une rotation à une vitesse de Omega.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from ipywidgets import interact
import ipywidgets as widgets

# Définir les paramètres de la rotation
Omega = 2.0 * np.pi / 86400   # Vitesse de rotation (rad/s)

# Définir les vecteurs de référence I et J
# Ces vecteurs sont indépendants du temps et donc de la rotation

I = np.array([1, 0])  # Vecteur I (x)
J = np.array([0, 1])  # Vecteur J (y)


# On définit un axe "temps" en secondes de 0 à 86400 (24 heures)
time = np.arange(0,14400)

# On va créer une fonction update() qui sera appelée à chaque étape de l'animation pour mettre à jour les positions des vecteurs I et J en fonction du temps.
def update(jt):

        fig, ax = plt.subplots (1, 2, figsize = (10, 6),  dpi = 150)
        
        # On crée une figure représentant les vecteurs de base I et J, absolument fixes.
        ax[0].arrow(0, 0, I[0], I[1], head_width=0.1, head_length=0.1, fc='blue', ec='blue', label='I')
        # On ajoute le nom du vecteur à côté de la tête de la flèche
        ax[0].text(I[0] + 0.1, I[1] + 0.1, r'$\vec{I}$', color='blue', fontsize=12)
        ax[0].arrow(0, 0, J[0], J[1], head_width=0.1, head_length=0.1, fc='blue', ec='blue', label='J')
        # On ajoute le nom du vecteur à côté de la tête de la flèche
        ax[0].text(J[0] + 0.1, J[1] + 0.1, r'$\vec{J}$', color='blue', fontsize=12)
        # On nomme et on écrit ce point O à côté de l'origine
        ax[0].scatter(0, 0, 10, color='blue', marker='o', label='Origine du repère fixe')
        ax[0].text(-0.2, -0.2, r'$O_a$', color='blue', fontsize=12)

        # On définit l' origine d'un repère tournant, dont les coordonnées sont R cos(Omega * t) et R sin(Omega * t
        R = 3.0  # Rayon de la rotation

        origin_rotating = np.array([R * np.cos(Omega * time), R * np.sin(Omega * time)])

       
        # On fixe le temps en fonction de l'argument jt qui a été passé dans la fonction
        t = time[jt]

        ax[0].scatter(origin_rotating[0, jt], origin_rotating[1, jt], 10, color='red', marker='o', label='Origine du repère tournant')
        # On nomme et on écrit ce point O_r à côté de l'origine
        ax[0].text(origin_rotating[0, jt] - 0.2, origin_rotating[1, jt] - 0.2, r'$O_r$', color='red', fontsize=12)


        # On trace la trajectoire déjà parcourue par l'origine du repère tournant depuis t = 0 jusqu'à t = time[jt]
        ax[0].plot(origin_rotating[0,0:jt], origin_rotating[1,0:jt], 'k:', label='Trajectoire du repère tournant')



        # On trace les vecteurs de base radial et tangentiel du repère tournant

        # Ces vecteurs sont des combinaisons linéaires de I et J:
        er = np.array([np.cos(Omega * t), np.sin(Omega * t)])  # Vecteur radial
        et = np.array([-np.sin(Omega * t), np.cos(Omega * t)]) # Vecteur tangentiel


        ax[0].arrow(origin_rotating[0, jt], origin_rotating[1, jt], er[0], er[1], head_width=0.1, head_length=0.1, fc='red', ec='red', label='er')
        ax[0].arrow(origin_rotating[0, jt], origin_rotating[1, jt], et[0], et[1], head_width=0.1, head_length=0.1, fc='red', ec='red', label='et')
        # On ajoute les noms des vecteurs à côté de la tête de la flèche
        ax[0].text(origin_rotating[0, jt] + er[0] + 0.1, origin_rotating[1, jt] + er[1] + 0.1, r'$\vec{e_r}$', color='red', fontsize=12)
        ax[0].text(origin_rotating[0, jt] + et[0] + 0.1, origin_rotating[1, jt] + et[1] + 0.1, r'$\vec{e_\theta}$', color='red', fontsize=12)



        # Enfin, on localise une particule de vitesse initiale (V, U) dans le repère tournant, c'est à dire de vitesse (V, U + Omega * R) dans le repère fixe, et on trace sa trajectoire.
        # On suppose que la particule n'est soumise à aucune force et donc se déplace en ligne droite à vitesse constante dans le repère fixe.
        
        V = 0.1e-3
        U = 0.3e-3
        
        
        v0 = np.array([V, U + Omega * R])  # Vitesse initiale de la particule

        # La variable particle_position donne les coordonnées dans le repère fixe de la trajectoire de la particule.

        particle_position = np.full((2, len(time)), np.nan)  # Initialiser les positions de la particule

        particle_position[0, :] = R + v0[0] * time 
        particle_position[1, :] = v0[1] * time  


        # On trace la trajectoire de la particule dans le repère fixe

        ax[0].plot(particle_position[0,0:jt], particle_position[1,0:jt], 'g--', label='Trajectoire de la particule')
        
        # On marque la particule avec un carré
        ax[0].scatter(particle_position[0, jt], particle_position[1, jt], 10, color='green', marker='s', label='Particule')  



        # Deuxième étape: on va maintenant représenter la trajectoire de la particule perçue depuis le repère tournant.
        
        # On se met dans le subplot de droite la même situation mais vue du repère tournant.

        # Le vecteur position de la particule dans le repère fixe est connue (ci-dessus)

        # On définit d'abord le vecteur position depuis l'origine du repère tournant, c'est à dire la différence entre 
        # le vecteur position calculé ci-dessus et le vecteur position de l'origine du repère tournant. 
        # On travaille toujours dans la base fixe.

        # On trace les vecteurs pour bien clarifier la situation
        # On trace le vecteur position de puis O_r:
        ax[0].arrow(0, 0, particle_position[0, jt], particle_position[1, jt], head_width=0.1, head_length=0.1, fc='lightblue', zorder = -1000, ec='lightblue', label='Vecteur position de la particule dans la base fixe')

        # On trace le vecteur position depuis O_a
        ax[0].arrow(0, 0, origin_rotating[0, jt], origin_rotating[1, jt], head_width=0.1, head_length=0.1, fc='lightblue', ec='lightblue', zorder = -1000, label='Vecteur position de l\'origine du repère tournant dans la base fixe')

        # On trace le vecteur position de l'origine du repère tournant depuis O_a
        ax[0].arrow(0, 0, origin_rotating[0, jt], origin_rotating[1, jt], head_width=0.1, head_length=0.1, fc=[1, 0.5, 0.5], ec=[1, 0.5, 0.5], zorder = -1000, label='Vecteur position de l\'origine du repère tournant dans la base fixe')

        # On trace enfin le vecteur position de la particule depuis O_r
        ax[0].arrow(origin_rotating[0, jt], origin_rotating[1, jt], particle_position[0, jt] - origin_rotating[0, jt], particle_position[1, jt] - origin_rotating[1, jt], head_width=0.1, head_length=0.1, fc=[0.5, 1, 0.5], ec=[0.5, 1, 0.5], zorder = -1000, label='Vecteur position de la particule depuis O_r dans la base fixe')

        # On trace un point en (3, 4) resté au sol, avec uns smiley
        bigDotX = 3
        bigDotY = 4
        ax[0].text(bigDotX, bigDotY, r'$\bigodot$', color='orange', fontsize=12)

        # Le vecteur position de la particule depuis l'origine du repère tournant est donnée, dans la base fixe, par
        tmp_1 = particle_position[0, :] - origin_rotating[0, :]  # Coordonnée x de la particule dans la base fixe, multiplie I
        tmp_2 = particle_position[1, :] - origin_rotating[1, :]  # Coordonnée y de la particule dans la base fixe, multiplie J


        # On projette le vecteur tmp_1 * I + tmp_2 * J sur les vecteurs er et et pour trouver les coordonnées de la particule dans le repère tournant.
        # Comme er = I * cos(Omega * t) + J * sin(Omega * t) et et = -I * sin(Omega * t) + J * cos(Omega * t), on trouve que:
        
        particle_position_er = tmp_1 * np.cos(Omega * time) + tmp_2 * np.sin(Omega * time)
        # et en et:
        particle_position_et = -tmp_1 * np.sin(Omega * time) + tmp_2 * np.cos(Omega * time)

        # On trace dans le second subplot les vecteurs er et et, ainsi que l'origine du repère tournant      
        ax[1].arrow(0, 0, 1, 0, head_width=0.1, head_length=0.1, fc='red', ec='red', label='er')
        ax[1].arrow(0, 0, 0, 1, head_width=0.1, head_length=0.1, fc='red', ec='red', label='et')
        # On ajoute le nom de l'origine du repère tournant à côté de l'origine
        ax[1].text(-0.4, -0.4, r'$O_r$', color='red', fontsize=12)


        ax[1].scatter(0, 0, 10, color='red', marker='o', label='Origine du repère tournant')
        ax[1].set_xlim(-2.5, 8.5)
        ax[1].set_ylim(-2.5, 8.5)
        ax[1].set_aspect('equal')
        ax[1].set_axisbelow(True)
        ax[1].grid()  

        # On trace la trajectoire de la particule dans le repère tournant   

        ax[1].plot(particle_position_er[0:jt], particle_position_et[0:jt ], 'g--', label='Trajectoire de la particule')
        # Marquer la particule avec un carré
        ax[1].scatter(particle_position_er[jt], particle_position_et[jt], 10, color='green', marker='s', label='Particule')   

        # On repère bigdot dans le repère tournant
        bigDotX_rot = (bigDotX - origin_rotating[0, jt]) * np.cos(Omega * time[jt]) + (bigDotY - origin_rotating[1, jt]) * np.sin(Omega * time[jt])
        bigDotY_rot = -(bigDotX - origin_rotating[0, jt]) * np.sin(Omega * time[jt]) + (bigDotY - origin_rotating[1, jt]) * np.cos(Omega * time[jt])
        ax[1].text(bigDotX_rot, bigDotY_rot, r'$\bigodot$', color='orange', fontsize=12)

        # On termine par du toilettage de figure
        ax[0].set_xlim(-0.5, 4.5)
        ax[0].set_ylim(-1.5, 4.5)
        #ax[0].legend(loc='upper left', fontsize = 6)
        ax[0].set_aspect('equal')
        ax[0].set_axisbelow(True)
        ax[0].grid()


        fig.tight_layout()


# Maintenant on crée  un slider pour faire varier le temps et observer l'animation.
interact(update, jt = (0, len(time)-1, 1))


interactive(children=(IntSlider(value=7199, description='jt', max=14399), Output()), _dom_classes=('widget-int…

<function __main__.update(jt)>